# MMEarth embedding exploration

This notebook extracts deterministic encoder embeddings from the pretraining validation split, checks their numerical quality and explores them with t-SNE. It also reports model parameters, feature-extraction FLOPs, and pretraining-archive pixels using the conventions from the CSMoE paper. t-SNE is descriptive only; downstream performance must be measured with task-specific probes.

## 1. Setup

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
config_path = repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
metrics_csv_path = None
max_samples = 5000
batch_size = 128
sample_seed = 42
token_source = 'pre_norm'
pooling = 'mean_fine'

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(repo_root))
from utils.analysis_utils import month_from_dates
from utils.extract_embeddings import (
    build_mmearth_dataset_from_config,
    build_model_from_dataset,
    build_raster_band_names,
    extract_embeddings,
    load_config,
    make_inference_dataloader,
    resolve_device,
)

## 2. Extract validation embeddings

The model is in evaluation mode, so top-k routing is deterministic. The same validity masks used during pretraining are passed to the encoder.

In [ ]:
config = load_config(str(config_path))
device = resolve_device()
dataset = build_mmearth_dataset_from_config(
    config, split='val', max_samples=max_samples, sample_seed=sample_seed
)
dataloader = make_inference_dataloader(dataset, batch_size=batch_size, num_workers=0)
model = build_model_from_dataset(config, dataset, str(checkpoint_path), device)
outputs = extract_embeddings(
    model,
    dataloader,
    build_raster_band_names(dataset),
    device,
    token_source=token_source,
    pooling=pooling,
)
embeddings = outputs['embeddings']
dates = outputs['dates']
print('embeddings:', embeddings.shape)
print('finite:', np.isfinite(embeddings).all())

## 3. Numerical checks

In [ ]:
embedding_norms = np.linalg.norm(embeddings, axis=1)
feature_std = embeddings.std(axis=0)
display(pd.Series(embedding_norms).describe().to_frame('embedding_norm'))
print('near-constant feature dimensions:', int((feature_std < 1e-6).sum()))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(embedding_norms, bins=40)
axes[0].set_title('Embedding norms')
axes[1].hist(feature_std, bins=40)
axes[1].set_title('Per-dimension standard deviation')
plt.tight_layout()
plt.show()

## 4. t-SNE

Standardization is applied only for visualization. Color by acquisition month where valid dates are available. Repeat with more than one t-SNE seed before treating a visual pattern as stable.

In [ ]:
scaled_embeddings = StandardScaler().fit_transform(embeddings)
perplexity = min(30, max(5, (len(embeddings) - 1) // 3))
projection = TSNE(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=sample_seed,
).fit_transform(scaled_embeddings)
months = month_from_dates(dates)
valid_months = months > 0
plt.figure(figsize=(9, 7))
if valid_months.any():
    scatter = plt.scatter(
        projection[valid_months, 0], projection[valid_months, 1],
        c=months[valid_months], cmap='turbo', s=8, alpha=0.75
    )
    plt.colorbar(scatter, label='Acquisition month')
else:
    plt.scatter(projection[:, 0], projection[:, 1], s=8, alpha=0.75)
plt.title(f't-SNE of {token_source}/{pooling} embeddings')
plt.xticks([])
plt.yticks([])
plt.show()

## 5. Pretraining metrics

In [ ]:
resolved_metrics_path = (
    Path(metrics_csv_path) if metrics_csv_path is not None
    else Path(config['training']['weight_path']) / f"training_metrics_{config['model']['size']}.csv"
)
print('metrics:', resolved_metrics_path)
if resolved_metrics_path.exists():
    metrics = pd.read_csv(resolved_metrics_path)
    display(metrics.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for column in ('train_loss', 'val_loss'):
        axes[0].plot(metrics['epoch'], metrics[column], label=column)
    for column in ('train_reconstruction_loss', 'val_reconstruction_loss'):
        axes[1].plot(metrics['epoch'], metrics[column], label=column)
    for column in ('train_moe_loss', 'val_moe_loss'):
        axes[2].plot(metrics['epoch'], metrics[column], label=column)
    for axis, title in zip(axes, ('Total loss', 'Reconstruction loss', 'Weighted MoE loss')):
        axis.set_title(title)
        axis.set_xlabel('epoch')
        axis.grid(alpha=0.25)
        axis.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Metrics CSV not found; set metrics_csv_path explicitly.')

## 6. Model scale: parameters, FLOPs, and PT PIX

This section follows [CSMoE Table I](https://arxiv.org/abs/2509.14104): FLOPs are measured for one batch-size-one forward feature-extraction pass from a 224 x 224 Sentinel-2 image using all S2 bands supported by this checkpoint. The MAE decoder is excluded from FLOPs because it is not used for feature extraction. `#P` reports the full checkpoint model, with the encoder/decoder split shown explicitly.

`PT PIX` is the number of spatial pixels in the actual training split: `training samples x height x width`. It excludes validation/test samples and does **not** multiply by bands, modalities, or epochs. CSMoE uses decimal units (`M=10^6`, `G=10^9`, `T=10^12`). FLOPs are counted with `fvcore`, where one multiply-add is one operation. This convention was validated by reproducing the TerraTorch Satlas result (`17.1201G` versus `17.12G` in CSMoE Table I). Unsupported element-wise operations are not included, matching the reference counter.

In [ ]:
def format_decimal_si(value):
    for scale, suffix in ((1e12, 'T'), (1e9, 'G'), (1e6, 'M'), (1e3, 'K')):
        if value >= scale:
            return f'{value / scale:.3f}{suffix}'
    return str(int(value))


total_parameters = sum(parameter.numel() for parameter in model.parameters())
encoder_parameters = sum(parameter.numel() for parameter in model.encoder.parameters())
decoder_parameters = total_parameters - encoder_parameters

full_train_dataset = build_mmearth_dataset_from_config(
    config, split='train', max_samples=None, sample_seed=sample_seed
)
train_samples = len(full_train_dataset)
configured_train_limit = config['training'].get('max_train_samples')
if configured_train_limit is not None:
    train_samples = min(train_samples, int(configured_train_limit))

pretrain_height = model.encoder.base_patch_height * model.encoder.patch_size
pretrain_width = model.encoder.base_patch_width * model.encoder.patch_size
pt_pixels = train_samples * pretrain_height * pretrain_width

scale_components = pd.DataFrame([
    {'quantity': '#P full MOEMAE', 'raw_value': total_parameters,
     'paper_unit': format_decimal_si(total_parameters)},
    {'quantity': '#P encoder', 'raw_value': encoder_parameters,
     'paper_unit': format_decimal_si(encoder_parameters)},
    {'quantity': '#P MAE decoder', 'raw_value': decoder_parameters,
     'paper_unit': format_decimal_si(decoder_parameters)},
    {'quantity': 'training samples', 'raw_value': train_samples,
     'paper_unit': format_decimal_si(train_samples)},
    {'quantity': 'PT PIX', 'raw_value': pt_pixels,
     'paper_unit': format_decimal_si(pt_pixels)},
])
display(scale_components)
print(f'Pretraining spatial size: {pretrain_height} x {pretrain_width}')
print(f'Decoder share of parameters: {100 * decoder_parameters / total_parameters:.2f}%')

In [ ]:
import torch
from torch import nn
from fvcore.nn import FlopCountAnalysis
import models.moe_mae as moe_mae_module


def traceable_apply_2d_rope(x, positions, theta=10000.0):
    # Same RoPE arithmetic without shape assertions that torch.jit misinterprets.
    if positions is None:
        return x
    if positions.ndim == 2:
        positions = positions.unsqueeze(0)
    rotary_dim = (x.shape[-1] // 4) * 4
    if rotary_dim == 0:
        return x
    axis_dim = rotary_dim // 2
    frequency = torch.arange(0, axis_dim, 2, device=x.device, dtype=torch.float32)
    frequency = theta ** (-frequency / axis_dim)
    coordinates = positions.to(device=x.device, dtype=torch.float32)
    rotated_axes = []
    for axis_index in range(2):
        axis = x[..., axis_index * axis_dim : (axis_index + 1) * axis_dim]
        angles = coordinates[..., axis_index].unsqueeze(-1) * frequency
        angles = angles.repeat_interleave(2, dim=-1).unsqueeze(1)
        rotated_axes.append(
            axis * angles.cos().to(x.dtype)
            + moe_mae_module._rotate_pairs(axis) * angles.sin().to(x.dtype)
        )
    return torch.cat([*rotated_axes, x[..., rotary_dim:]], dim=-1)


class EncoderFeatureCounter(nn.Module):
    def __init__(self, source_model, raster_modalities, include_metadata):
        super().__init__()
        self.source_model = source_model
        self.raster_modalities = tuple(raster_modalities)
        self.metadata_modalities = (
            tuple(source_model.encoder.metadata_dims) if include_metadata else ()
        )

    def forward(self, *inputs):
        raster_count = len(self.raster_modalities)
        raster_dict = dict(zip(self.raster_modalities, inputs[:raster_count]))
        meta_dict = dict(zip(self.metadata_modalities, inputs[raster_count:])) or None
        features = self.source_model.forward_features(
            raster_dict=raster_dict,
            raster_valid_masks={
                name: torch.ones_like(tensor, dtype=torch.bool)
                for name, tensor in raster_dict.items()
            },
            raster_band_names={
                name: list(self.source_model.encoder.input_band_names[name])
                for name in self.raster_modalities
            },
            meta_dict=meta_dict,
            meta_valid_masks=(
                {name: torch.ones_like(tensor, dtype=torch.bool)
                 for name, tensor in meta_dict.items()} if meta_dict else None
            ),
            stochastic_routing=False,
        )
        return features['pre_norm_fine_tokens']


def count_encoder_feature_flops(image_size, raster_modalities, include_metadata):
    inputs = [
        torch.zeros(
            1, len(model.encoder.input_band_names[name]), image_size, image_size,
            device=device,
        )
        for name in raster_modalities
    ]
    if include_metadata:
        inputs.extend(
            torch.zeros(1, width, device=device)
            for width in model.encoder.metadata_dims.values()
        )
    wrapper = EncoderFeatureCounter(model, raster_modalities, include_metadata).eval()
    original_rope = moe_mae_module.apply_2d_rope
    moe_mae_module.apply_2d_rope = traceable_apply_2d_rope
    try:
        analysis = FlopCountAnalysis(wrapper, tuple(inputs))
        analysis.unsupported_ops_warnings(False)
        analysis.uncalled_modules_warnings(False)
        analysis.tracer_warnings('none')
        return int(analysis.total())
    finally:
        moe_mae_module.apply_2d_rope = original_rope


csmoe_s2_flops = count_encoder_feature_flops(
    image_size=224, raster_modalities=('sentinel2',), include_metadata=False
)
native_s2_flops = count_encoder_feature_flops(
    image_size=pretrain_height, raster_modalities=('sentinel2',),
    include_metadata=False,
)
native_multimodal_flops = count_encoder_feature_flops(
    image_size=pretrain_height,
    raster_modalities=tuple(model.encoder.input_names),
    include_metadata=True,
)

headline_report = pd.DataFrame([
    {
        'model': f"MoE-MAE-{config['model']['size']}",
        '#P': format_decimal_si(total_parameters),
        'FLOPs (S2 224x224)': format_decimal_si(csmoe_s2_flops),
        'FLOPs (S2 64x64)': format_decimal_si(native_s2_flops),
        'PT PIX': format_decimal_si(pt_pixels),
    }
])
display(headline_report)
print('CSMoE-convention S2 224x224 FLOPs:', csmoe_s2_flops)
print('Native S2 64x64 FLOPs:', native_s2_flops)
print(
    f'Native {pretrain_height}x{pretrain_width} all-raster + metadata FLOPs: '
    f'{format_decimal_si(native_multimodal_flops)} ({native_multimodal_flops:,})'
)

## 7. Takeaways

After execution, record whether embeddings are finite and non-degenerate, whether t-SNE structure is stable across seeds, and whether visible clusters are explained by acquisition month. Report `#P`, S2-at-224 FLOPs, and `PT PIX` together with the conventions stated above so comparisons remain reproducible. Do not use t-SNE separation as a substitute for linear-probe or retrieval metrics.